# Agent Evaluation with MLflow

This notebook demonstrates how to evaluate the Public Assistant agent using MLflow's evaluation framework.

We support two evaluation modes:
- **Simple mode**: Fast, deterministic checks (no LLM calls)
- **LLM-as-a-Judge mode**: Full evaluation with LLM judges for tool correctness, safety, and relevance

## Prerequisites

1. Complete the project setup (`make setup`)
2. Configure your `.env` file with:
   - `MLFLOW_TRACKING_URI`: MLflow server URL
   - `MLFLOW_EXPERIMENT_NAME`: Experiment name
   - `LLM_BASE_URL`: LLM endpoint URL
   - `LLM_MODEL_CAPABLE`: Model name for the judge
3. Get an MLflow tracking token:
   ```bash
   export MLFLOW_TRACKING_TOKEN=$(oc whoami --show-token)
   ```

## Setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Suppress warnings
warnings.filterwarnings("ignore")

# Load environment variables
from dotenv import load_dotenv
load_dotenv(override=False)

# Add packages/api to path
api_path = Path.cwd().parent / "packages" / "api"
if str(api_path) not in sys.path:
    sys.path.insert(0, str(api_path))

# Verify required environment variables
required_vars = ["MLFLOW_TRACKING_URI", "MLFLOW_EXPERIMENT_NAME"]
missing = [v for v in required_vars if not os.environ.get(v)]
if missing:
    raise ValueError(f"Missing required environment variables: {', '.join(missing)}")

print(f"MLflow URI: {os.environ.get('MLFLOW_TRACKING_URI')}")
print(f"Experiment: {os.environ.get('MLFLOW_EXPERIMENT_NAME')}")

## Configure MLflow

In [ ]:
import logging
import mlflow

# Suppress MLflow warnings
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Configure MLflow
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

experiment_name = os.environ["MLFLOW_EXPERIMENT_NAME"]
if not experiment_name.endswith("-eval"):
    experiment_name = f"{experiment_name}-eval"

mlflow.set_experiment(experiment_name)
mlflow.langchain.autolog()

print(f"Experiment set: {experiment_name}")

## 1. Create Evaluation Dataset

First, we create a dataset on the MLflow server. Each test case has:
- `inputs`: The user message to send to the agent
- `expectations`: Expected behavior including:
  - `expected_answer`: Keyword that should appear in the response
  - `expected_tool_calls`: Tools the agent should call (MLflow format)
  - `expected_topics`: Topics the response should cover
  - `forbidden_content`: Content that should NOT appear

In [ ]:
from mlflow.genai.datasets import create_dataset

# Create a new dataset on the MLflow server
dataset_name = "public_assistant_eval"
dataset = create_dataset(
    name=dataset_name,
    tags={"stage": "validation", "version": "1", "agent": "public-assistant"},
)

# Define test cases for the Public Assistant agent
test_cases = [
    # Product Information - Simple Questions
    {
        "inputs": {"user_message": "What loan products do you offer?"},
        "expectations": {
            "expected_answer": "30-year",
            "expected_tool_calls": [{"name": "product_info"}],
            "expected_topics": ["fixed", "FHA", "VA"],
            "forbidden_content": [],
        },
    },
    {
        "inputs": {"user_message": "Tell me about FHA loans"},
        "expectations": {
            "expected_answer": "FHA",
            "expected_tool_calls": [{"name": "product_info"}],
            "expected_topics": ["down payment"],
            "forbidden_content": [],
        },
    },
    {
        "inputs": {"user_message": "What is a VA loan?"},
        "expectations": {
            "expected_answer": "VA",
            "expected_tool_calls": [{"name": "product_info"}],
            "expected_topics": ["veteran", "military"],
            "forbidden_content": [],
        },
    },
    {
        "inputs": {"user_message": "Compare fixed vs adjustable rate mortgages"},
        "expectations": {
            "expected_answer": "fixed",
            "expected_tool_calls": [{"name": "product_info"}],
            "expected_topics": ["ARM", "rate"],
            "forbidden_content": [],
        },
    },
    # Affordability - Complete info for tool to be called
    {
        "inputs": {
            "user_message": "I make $100,000 a year with $500 monthly debts and $20,000 for down payment. How much house can I afford?"
        },
        "expectations": {
            "expected_answer": "afford",
            "expected_tool_calls": [{"name": "affordability_calc"}],
            "expected_topics": ["loan", "payment"],
            "forbidden_content": ["approved", "guaranteed"],
        },
    },
    {
        "inputs": {
            "user_message": "What would my monthly payment be on a $300,000 loan at 6.5% for 30 years?"
        },
        "expectations": {
            "expected_answer": "payment",
            "expected_tool_calls": [{"name": "affordability_calc"}],
            "expected_topics": ["monthly", "interest"],
            "forbidden_content": [],
        },
    },
]

# Add test cases to the dataset
dataset = dataset.merge_records(test_cases)
print(f"Dataset created: {dataset.dataset_id}")
print(f"Test cases: {len(test_cases)}")

## 2. View the Dataset

Let's inspect the test cases we just created.

In [ ]:
# View the dataset contents
print(f"Dataset: {dataset_name}")
print(f"Total test cases: {len(test_cases)}\n")

for i, case in enumerate(test_cases, 1):
    msg = case['inputs']['user_message']
    display_msg = msg[:60] + "..." if len(msg) > 60 else msg
    print(f"{i}. {display_msg}")
    print(f"   Expected keyword: {case['expectations']['expected_answer']}")
    print(f"   Expected tools: {[t['name'] for t in case['expectations'].get('expected_tool_calls', [])]}")
    print()

## 3. Predictor Function

The predictor wraps the agent invocation for MLflow's evaluate function.

In [ ]:
from predictors import get_predictor

# Get the predictor for public-assistant
predict_fn = get_predictor("public-assistant")

# Test it with a simple query
test_response = predict_fn("What loan products do you offer?")
print(f"Test response preview: {test_response[:200]}...")

## 4. Scorers

We have two types of scorers:

### Simple Scorers (no LLM calls)
- `contains_expected`: Checks if expected keyword appears in response
- `has_numeric_result`: Checks if response contains numbers (for calculations)
- `response_length`: Ensures adequate response length

### LLM-as-a-Judge Scorers
- `ToolCallCorrectness`: Did the agent call the right tools?
- `ToolCallEfficiency`: Were tool calls minimal and efficient?
- `RelevanceToQuery`: Is the response relevant to the question?
- `Safety`: Is the response safe and appropriate?
- `Guidelines`: Does response follow mortgage assistant guidelines?

In [ ]:
from scorers.custom_scorers import contains_expected, has_numeric_result, response_length

# Simple scorers (always available)
simple_scorers = [
    contains_expected,
    has_numeric_result,
    response_length,
]

print("Simple scorers loaded:")
for s in simple_scorers:
    print(f"  - {s.name}")

In [ ]:
# LLM-as-a-Judge scorers (requires LLM endpoint)
from mlflow.genai.scorers import (
    Guidelines,
    RelevanceToQuery,
    Safety,
    ToolCallCorrectness,
    ToolCallEfficiency,
)

# Configure judge model
base_url = os.environ.get("LLM_BASE_URL")
model_name = os.environ.get("LLM_MODEL_CAPABLE")

if base_url and model_name:
    os.environ["OPENAI_API_BASE"] = base_url
    os.environ["OPENAI_BASE_URL"] = base_url
    judge_model = f"openai:/{model_name}"
    print(f"Judge model: {judge_model}")
    
    # Create LLM judge scorers
    llm_scorers = [
        ToolCallCorrectness(model=judge_model, should_exact_match=True),
        ToolCallEfficiency(model=judge_model),
        RelevanceToQuery(model=judge_model),
        Safety(model=judge_model),
        Guidelines(
            name="mortgage_guidelines",
            guidelines=[
                "Response should be helpful about mortgage products",
                "Response should NOT promise specific rates or pre-approval",
                "Response should use professional, clear language",
            ],
            model=judge_model,
        ),
    ]
    print(f"LLM judge scorers: {len(llm_scorers)}")
else:
    llm_scorers = []
    print("LLM judge not configured - set LLM_BASE_URL and LLM_MODEL_CAPABLE")

## 5. Run Simple Evaluation

Fast evaluation using only deterministic scorers.

In [ ]:
# Run simple evaluation using the dataset
print("Running simple evaluation...")
print(f"Dataset: {len(test_cases)} examples")
print(f"Scorers: {len(simple_scorers)}")
print()

simple_results = mlflow.genai.evaluate(
    data=dataset,
    predict_fn=predict_fn,
    scorers=simple_scorers,
)

print("\nSimple Evaluation Results:")
print("-" * 40)
for metric, value in sorted(simple_results.metrics.items()):
    if isinstance(value, float):
        print(f"  {metric}: {value:.2%}")
    else:
        print(f"  {metric}: {value}")

## 6. Run Full LLM-as-a-Judge Evaluation

Complete evaluation including LLM judges for tool correctness, safety, and relevance.

In [ ]:
if llm_scorers:
    all_scorers = simple_scorers + llm_scorers
    
    print("Running LLM-as-a-Judge evaluation...")
    print(f"Dataset: {len(test_cases)} examples")
    print(f"Scorers: {len(all_scorers)} ({len(llm_scorers)} LLM judges)")
    print()
    
    full_results = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=predict_fn,
        scorers=all_scorers,
    )
    
    print("\nFull Evaluation Results:")
    print("-" * 40)
    for metric, value in sorted(full_results.metrics.items()):
        if isinstance(value, float):
            print(f"  {metric}: {value:.2%}")
        else:
            print(f"  {metric}: {value}")
else:
    print("Skipping LLM-as-a-Judge evaluation (not configured)")
    print("Set LLM_BASE_URL and LLM_MODEL_CAPABLE in your .env file")

## 7. View Results in MLflow

Navigate to the MLflow UI to see detailed results:

1. Open your MLflow tracking URI in a browser
2. Go to **Experiments** > your-experiment-eval
3. Click on **Evaluation Runs** tab
4. Select your run to view per-trace assessments
5. Enable **All Assessments** in the Columns dropdown to see LLM judge scores

You can also view the dataset under **Experiments** > Default > **Datasets**

In [ ]:
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
print(f"View results at: {tracking_uri}/#/experiments")
print(f"Dataset ID: {dataset.dataset_id}")

## 8. Human Feedback

After automated evaluations, share results with subject matter experts:

### Add New Feedback
1. Navigate to your experiment in MLflow UI
2. Click **Add Assessment** button
3. Select **Feedback** from Assessment Type
4. Enter name, value, and rationale
5. Click **Create**

### Update Existing Feedback
1. Find the feedback entry
2. Click the hamburger menu (⋮)
3. Select **Edit**
4. Modify and save

## CLI Usage

You can also run evaluations from the command line:

```bash
# Simple mode (fast, no LLM judge)
MLFLOW_TRACKING_TOKEN=$(oc whoami --show-token) uv run python -m evaluations.run_agent_eval --mode simple

# LLM-as-a-Judge mode (full evaluation)
MLFLOW_TRACKING_TOKEN=$(oc whoami --show-token) uv run python -m evaluations.run_agent_eval --mode llm-judge
```